# 08 · Random Forest

Hundreds of decision trees, each grown on a bootstrap sample of women and a random share of the predictors at every split, averaged. It needs no scaling and finds interactions by itself.

**Preparation.** Numbers: median fill. Categories: the most frequent level fills a missing value, then each level becomes an integer code the trees split on.

**Tuning** is the study's single procedure (notebook 04): Optuna's TPE sampler, seed 2026, a fixed
number of trials, each scored by survey-weighted average precision out of fold over the five
development folds, with survey weights rescaled to average one inside each fold. The search
space and the budget are printed below. After the search, the best settings are refitted on all
5,518 development women, and the held-out women are scored **once**. The operating threshold
(for sensitivity and specificity) is chosen on the development predictions only.

The model is tuned separately on every predictor set at every rung: 12 fits, each saved to
`results/<set>/<rung>/Random Forest.npz` with its out-of-fold and held-out predictions and its
settings. A cell that is already saved is skipped, so the notebook can be stopped and resumed.

In [ ]:
# Settings for this notebook
QUICK = False        # True: a smoke run with 3 search trials per model; numbers will not match the study
FOLLOW_STUDY = True  # when a re-derived choice differs from the study's, carry the study's forward

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib, tuning

MODEL = "Random Forest"
FOLD_JOBS = 5   # folds fitted in parallel; lower it on a machine with few cores

In [ ]:
# The search space: each setting, its range, and whether it is searched on a log scale.
from anc8 import models as zoo
class Space:
    def __init__(self): self.rows = []
    def suggest_float(self, n, lo, hi, log=False, step=None): self.rows.append((n, lo, hi, "log" if log else "")); return lo
    def suggest_int(self, n, lo, hi, step=1, log=False): self.rows.append((n, lo, hi, f"step {step}")); return lo
    def suggest_categorical(self, n, ch): self.rows.append((n, ch, "", "one of")); return ch[0]
sp = Space(); zoo.suggest(sp, MODEL)
print(pd.DataFrame(sp.rows, columns=["setting", "from", "to", ""]).to_string(index=False))
print(f"\n{tuning.TRIALS[MODEL]} trials per predictor set and rung")

In [ ]:
for s in lib.SETS:
    for r in lib.RUNGS:
        job = lib.Job("tune", s, r, MODEL, MODEL)
        if lib.is_done(P.results, job):
            print(f"{job.id}: already done")
            continue
        lib.run(job, P.results, fold_jobs=FOLD_JOBS)

## Results, and the check against the study

In [ ]:
res = repro.compare_cells(P.results, [MODEL])
print(res.pivot_table(index="set", columns="rung", values="held_AP").round(3).to_string())
repro.report()